# ✈️ Uçuş Fiyat Tahmini: Keşifçi Veri Analizi ve Özellik Mühendisliği (Flight Price EDA)

Bu projede, Hindistan içi uçuş verilerini içeren gerçek dünya veri seti üzerinde **kapsamlı bir Keşifçi Veri Analizi (EDA)** ve **Uçtan Uca Özellik Mühendisliği (Feature Engineering)** boru hattı inşa edilmektedir.

---

### 🎯 Adım Adım İşlem Haritası:
1. **Zaman Serisi & Tarih Mühendisliği:** `Date_of_Journey` üzerinden Gün, Ay ve Yıl sütunlarının türetilmesi.
2. **Saat/Dakika Ayrıştırması:** `Dep_Time` ve `Arrival_Time` sütunlarının saat ve dakika cinsine ayrılması.
3. **Uçuş Süresi (Duration) Dönüşümü:** Karmaşık saat ve dakika string ifadelerinin toplam dakika cinsine çevrilmesi.
4. **Aktarma Sayısı (Total_Stops) Kodlaması:** Ordinal haritalama (`non-stop -> 0`, `1 stop -> 1` vb.).
5. **Kategorik Değişken Kodlaması (Encoding):**
   - Havayolu (`Airline`) için **One-Hot Encoding**.
   - Kalkış, Varış ve Ekstra Bilgi için **Target Guided Mean Encoding**.


In [1]:
#Dengesiz veri kümeleri(imbalanced dataset)
# UpSampling
# DownSampling
# SMOTE

# Encoding(Kategorik verileri nasıl nümerik veriler haline getirebiliriz)
# LabelEncoding-OrdinalEncoding
# One Hot Encoding
# Target Guided Encoding (Atıştırma: 300, bakliyat: 200)

In [2]:
# Explorative Data Analysis(EDA)(Veri Kaşifliği)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_flight_price = pd.read_excel("flight_price.xlsx")
df_flight_price.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [3]:
df_flight_price.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Airline          10683 non-null  str  
 1   Date_of_Journey  10683 non-null  str  
 2   Source           10683 non-null  str  
 3   Destination      10683 non-null  str  
 4   Route            10682 non-null  str  
 5   Dep_Time         10683 non-null  str  
 6   Arrival_Time     10683 non-null  str  
 7   Duration         10683 non-null  str  
 8   Total_Stops      10682 non-null  str  
 9   Additional_Info  10683 non-null  str  
 10  Price            10683 non-null  int64
dtypes: int64(1), str(10)
memory usage: 918.2 KB


In [4]:
df_flight_price["Airline"].unique()

<StringArray>
[                           'IndiGo',                         'Air India',
                       'Jet Airways',                          'SpiceJet',
                 'Multiple carriers',                             'GoAir',
                           'Vistara',                          'Air Asia',
           'Vistara Premium economy',              'Jet Airways Business',
 'Multiple carriers Premium economy',                            'Trujet']
Length: 12, dtype: str

In [5]:
# Veri kümesinde numeric veya kategorik olmayan veri gruplarıda bulunabilir
# Date_of_Journey

df_years = pd.to_datetime(df_flight_price["Date_of_Journey"], dayfirst= True).dt.year
df_monts = pd.to_datetime(df_flight_price["Date_of_Journey"], dayfirst= True).dt.month
df_days = pd.to_datetime(df_flight_price["Date_of_Journey"], dayfirst= True).dt.day

df_flight_price["days"] = df_days
df_flight_price["month"] = df_monts
df_flight_price["years"] = df_years


df_flight_price.drop("Date_of_Journey", axis = 1, inplace = True)
df_flight_price.head()

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,days,month,years
0,IndiGo,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,24,3,2019
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662,1,5,2019
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882,9,6,2019
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218,12,5,2019
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302,1,3,2019


In [6]:
df_flight_price.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Airline          10683 non-null  str  
 1   Source           10683 non-null  str  
 2   Destination      10683 non-null  str  
 3   Route            10682 non-null  str  
 4   Dep_Time         10683 non-null  str  
 5   Arrival_Time     10683 non-null  str  
 6   Duration         10683 non-null  str  
 7   Total_Stops      10682 non-null  str  
 8   Additional_Info  10683 non-null  str  
 9   Price            10683 non-null  int64
 10  days             10683 non-null  int32
 11  month            10683 non-null  int32
 12  years            10683 non-null  int32
dtypes: int32(3), int64(1), str(9)
memory usage: 959.9 KB


In [7]:
df_flight_price["Arrival_Time"].value_counts()
df_hour_minute = df_flight_price["Arrival_Time"].apply(lambda element: element.split(" ")[0])

df_hours = df_hour_minute.str.split(":").str[0]
df_minutes = df_hour_minute.str.split(":").str[1]

df_flight_price["Arrival_hour"] = df_hours.astype(int)
df_flight_price["Arrival_minutes"] = df_minutes.astype(int)


df_flight_price.drop("Arrival_Time", axis = 1, inplace = True)
df_flight_price.head()

,Airline,Source,Destination,Route,Dep_Time,Duration,Total_Stops,Additional_Info,Price,days,month,years,Arrival_hour,Arrival_minutes
0,IndiGo,Banglore,New Delhi,BLR → DEL,22:20,2h 50m,non-stop,No info,3897,24,3,2019,1,10
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,7h 25m,2 stops,No info,7662,1,5,2019,13,15
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,19h,2 stops,No info,13882,9,6,2019,4,25
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,18:05,5h 25m,1 stop,No info,6218,12,5,2019,23,30
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,16:50,4h 45m,1 stop,No info,13302,1,3,2019,21,35


In [8]:
df_flight_price["Dep_Time"].value_counts()

df_flight_price["Dep_hour"] = df_flight_price["Dep_Time"].str.split(":").str[0].astype(int)
df_flight_price["Dep_minutes"] = df_flight_price["Dep_Time"].str.split(":").str[1].astype(int)

df_flight_price.drop("Dep_Time", axis = 1, inplace= True)
df_flight_price.head()



,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,days,month,years,Arrival_hour,Arrival_minutes,Dep_hour,Dep_minutes
0,IndiGo,Banglore,New Delhi,BLR → DEL,2h 50m,non-stop,No info,3897,24,3,2019,1,10,22,20
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,7h 25m,2 stops,No info,7662,1,5,2019,13,15,5,50
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,19h,2 stops,No info,13882,9,6,2019,4,25,9,25
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,5h 25m,1 stop,No info,6218,12,5,2019,23,30,18,5
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,4h 45m,1 stop,No info,13302,1,3,2019,21,35,16,50


In [9]:
df_duration_hours = df_flight_price["Duration"].str.split(" ").str[0]
df_duration_minutes = df_flight_price["Duration"].str.split(" ").str[1]

df_duration_hours = df_duration_hours.str.replace("h","")
df_duration_hours[6474] = 0
df_duration_hours = df_duration_hours.astype(int)


df_duration_minutes = df_duration_minutes.str.replace("m","")
df_duration_minutes= df_duration_minutes.fillna(0)
df_duration_minutes.isnull().sum()
df_duration_minutes = df_duration_minutes.astype(int)
df_duration_minutes

df_flight_price["Duration_minutes"] = df_duration_hours * 60 + df_duration_minutes
df_flight_price.drop("Duration", axis = 1 , inplace= True)
df_flight_price.head()

,Airline,Source,Destination,Route,Total_Stops,Additional_Info,Price,days,month,years,Arrival_hour,Arrival_minutes,Dep_hour,Dep_minutes,Duration_minutes
0,IndiGo,Banglore,New Delhi,BLR → DEL,non-stop,No info,3897,24,3,2019,1,10,22,20,170
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,2 stops,No info,7662,1,5,2019,13,15,5,50,445
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,2 stops,No info,13882,9,6,2019,4,25,9,25,1140
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,1 stop,No info,6218,12,5,2019,23,30,18,5,325
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,1 stop,No info,13302,1,3,2019,21,35,16,50,285


In [10]:
df_flight_price.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Airline           10683 non-null  str  
 1   Source            10683 non-null  str  
 2   Destination       10683 non-null  str  
 3   Route             10682 non-null  str  
 4   Total_Stops       10682 non-null  str  
 5   Additional_Info   10683 non-null  str  
 6   Price             10683 non-null  int64
 7   days              10683 non-null  int32
 8   month             10683 non-null  int32
 9   years             10683 non-null  int32
 10  Arrival_hour      10683 non-null  int64
 11  Arrival_minutes   10683 non-null  int64
 12  Dep_hour          10683 non-null  int64
 13  Dep_minutes       10683 non-null  int64
 14  Duration_minutes  10683 non-null  int64
dtypes: int32(3), int64(6), str(6)
memory usage: 1.1 MB


In [11]:
df_flight_price["Total_Stops"].mode()
df_flight_price["Total_Stops"] = df_flight_price["Total_Stops"].fillna(df_flight_price["Total_Stops"].mode()[0])
df_flight_price["Total_Stops"].unique()


<StringArray>
['non-stop', '2 stops', '1 stop', '3 stops', '4 stops']
Length: 5, dtype: str

In [12]:
df_flight_price["Total_Stops"].value_counts()

Total_Stops
1 stop      5626
non-stop    3491
2 stops     1520
3 stops       45
4 stops        1
Name: count, dtype: int64

In [13]:
stops = {
    "non-stop": 0,
    "1 stop": 1,
    "2 stops": 2,
    "3 stops": 3,
    "4 stops": 4
}
df_flight_price["Total_Stops"] = df_flight_price["Total_Stops"].map(stops)
df_flight_price

,Airline,Source,Destination,Route,Total_Stops,Additional_Info,Price,days,month,years,Arrival_hour,Arrival_minutes,Dep_hour,Dep_minutes,Duration_minutes
0,IndiGo,Banglore,New Delhi,BLR → DEL,0,No info,3897,24,3,2019,1,10,22,20,170
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,2,No info,7662,1,5,2019,13,15,5,50,445
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,2,No info,13882,9,6,2019,4,25,9,25,1140
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,1,No info,6218,12,5,2019,23,30,18,5,325
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,1,No info,13302,1,3,2019,21,35,16,50,285
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,Air Asia,Kolkata,Banglore,CCU → BLR,0,No info,4107,9,4,2019,22,25,19,55,150
10679,Air India,Kolkata,Banglore,CCU → BLR,0,No info,4145,27,4,2019,23,20,20,45,155
10680,Jet Airways,Banglore,Delhi,BLR → DEL,0,No info,7229,27,4,2019,11,20,8,20,180
10681,Vistara,Banglore,New Delhi,BLR → DEL,0,No info,12648,1,3,2019,14,10,11,30,160


In [14]:
df_flight_price.drop("Route", axis = 1, inplace= True)
df_flight_price.head()

,Airline,Source,Destination,Total_Stops,Additional_Info,Price,days,month,years,Arrival_hour,Arrival_minutes,Dep_hour,Dep_minutes,Duration_minutes
0,IndiGo,Banglore,New Delhi,0,No info,3897,24,3,2019,1,10,22,20,170
1,Air India,Kolkata,Banglore,2,No info,7662,1,5,2019,13,15,5,50,445
2,Jet Airways,Delhi,Cochin,2,No info,13882,9,6,2019,4,25,9,25,1140
3,IndiGo,Kolkata,Banglore,1,No info,6218,12,5,2019,23,30,18,5,325
4,IndiGo,Banglore,New Delhi,1,No info,13302,1,3,2019,21,35,16,50,285


In [15]:
df_flight_price.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Airline           10683 non-null  str  
 1   Source            10683 non-null  str  
 2   Destination       10683 non-null  str  
 3   Total_Stops       10683 non-null  int64
 4   Additional_Info   10683 non-null  str  
 5   Price             10683 non-null  int64
 6   days              10683 non-null  int32
 7   month             10683 non-null  int32
 8   years             10683 non-null  int32
 9   Arrival_hour      10683 non-null  int64
 10  Arrival_minutes   10683 non-null  int64
 11  Dep_hour          10683 non-null  int64
 12  Dep_minutes       10683 non-null  int64
 13  Duration_minutes  10683 non-null  int64
dtypes: int32(3), int64(7), str(4)
memory usage: 1.0 MB


In [16]:
df_flight_price["Airline"].unique()

<StringArray>
[                           'IndiGo',                         'Air India',
                       'Jet Airways',                          'SpiceJet',
                 'Multiple carriers',                             'GoAir',
                           'Vistara',                          'Air Asia',
           'Vistara Premium economy',              'Jet Airways Business',
 'Multiple carriers Premium economy',                            'Trujet']
Length: 12, dtype: str

In [17]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()
encoded = encoder.fit_transform(df_flight_price[["Airline"]]).toarray()
encoded_df = pd.DataFrame(encoded, columns= encoder.get_feature_names_out())
df_flight_price = pd.concat([df_flight_price, encoded_df], axis = 1)
df_flight_price.head()


,Airline,Source,Destination,Total_Stops,Additional_Info,Price,days,month,years,Arrival_hour,...,Airline_GoAir,Airline_IndiGo,Airline_Jet Airways,Airline_Jet Airways Business,Airline_Multiple carriers,Airline_Multiple carriers Premium economy,Airline_SpiceJet,Airline_Trujet,Airline_Vistara,Airline_Vistara Premium economy
0,IndiGo,Banglore,New Delhi,0,No info,3897,24,3,2019,1,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Air India,Kolkata,Banglore,2,No info,7662,1,5,2019,13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Jet Airways,Delhi,Cochin,2,No info,13882,9,6,2019,4,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,IndiGo,Kolkata,Banglore,1,No info,6218,12,5,2019,23,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,IndiGo,Banglore,New Delhi,1,No info,13302,1,3,2019,21,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
df_flight_price.drop("Airline", axis = 1, inplace= True)

In [19]:
df_flight_price.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 25 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Source                                     10683 non-null  str    
 1   Destination                                10683 non-null  str    
 2   Total_Stops                                10683 non-null  int64  
 3   Additional_Info                            10683 non-null  str    
 4   Price                                      10683 non-null  int64  
 5   days                                       10683 non-null  int32  
 6   month                                      10683 non-null  int32  
 7   years                                      10683 non-null  int32  
 8   Arrival_hour                               10683 non-null  int64  
 9   Arrival_minutes                            10683 non-null  int64  
 10  Dep_hour                         

In [20]:
df_flight_price["Destination"].unique()

<StringArray>
['New Delhi', 'Banglore', 'Cochin', 'Kolkata', 'Delhi', 'Hyderabad']
Length: 6, dtype: str

In [21]:
# target guided encoding 
price_destination_result = df_flight_price.groupby("Destination")["Price"].mean().astype(int).to_dict()

df_flight_price["Destination"] = df_flight_price["Destination"].map(price_destination_result)
df_flight_price.head()

,Source,Destination,Total_Stops,Additional_Info,Price,days,month,years,Arrival_hour,Arrival_minutes,...,Airline_GoAir,Airline_IndiGo,Airline_Jet Airways,Airline_Jet Airways Business,Airline_Multiple carriers,Airline_Multiple carriers Premium economy,Airline_SpiceJet,Airline_Trujet,Airline_Vistara,Airline_Vistara Premium economy
0,Banglore,11917,0,No info,3897,24,3,2019,1,10,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Kolkata,9158,2,No info,7662,1,5,2019,13,15,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Delhi,10539,2,No info,13882,9,6,2019,4,25,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Kolkata,9158,1,No info,6218,12,5,2019,23,30,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Banglore,11917,1,No info,13302,1,3,2019,21,35,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
df_flight_price.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 25 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Source                                     10683 non-null  str    
 1   Destination                                10683 non-null  int64  
 2   Total_Stops                                10683 non-null  int64  
 3   Additional_Info                            10683 non-null  str    
 4   Price                                      10683 non-null  int64  
 5   days                                       10683 non-null  int32  
 6   month                                      10683 non-null  int32  
 7   years                                      10683 non-null  int32  
 8   Arrival_hour                               10683 non-null  int64  
 9   Arrival_minutes                            10683 non-null  int64  
 10  Dep_hour                         

In [24]:
df_flight_price["Source"].unique()

<StringArray>
['Banglore', 'Kolkata', 'Delhi', 'Chennai', 'Mumbai']
Length: 5, dtype: str

In [25]:
df_flight_price["Additional_Info"].unique()

<StringArray>
[                     'No info',  'In-flight meal not included',
 'No check-in baggage included',              '1 Short layover',
                      'No Info',               '1 Long layover',
              'Change airports',               'Business class',
               'Red-eye flight',               '2 Long layover']
Length: 10, dtype: str

In [26]:
price_destination_result = df_flight_price.groupby("Additional_Info")["Price"].mean().astype(int).to_dict()
price_destination_result

{'1 Long layover': 22109,
 '1 Short layover': 26743,
 '2 Long layover': 26480,
 'Business class': 56811,
 'Change airports': 16800,
 'In-flight meal not included': 9471,
 'No Info': 8044,
 'No check-in baggage included': 3645,
 'No info': 9141,
 'Red-eye flight': 10873}